# L4b: Single Asset Geometric Brownian Motion Models
In this lecture, we introduce our first continuous-time model for asset prices, the Geometric Brownian Motion (GBM) model. This model is __widely__ used in finance because it is mathematically tractable: it has an exact solution, it can be simulated exactly, and it gives closed-form probabilities for the trade rules we developed last lecture.

> __Learning Objectives:__
>
> By the end of this lecture, you will be able to:
> 
> * **Model share prices with geometric Brownian motion:** State the GBM stochastic differential equation and the Wiener process that drives it, write the exact lognormal solution with its mean and variance, and use the exact one-step transition to simulate price paths by Monte Carlo.
> * **Estimate the drift and volatility parameters from data:** Derive the distribution of the one-step growth rate under GBM, estimate the mean growth rate and the volatility from a growth-rate series, convert the growth-rate standard deviation into the GBM volatility using the time step, and recover the arithmetic drift.
> * **Test GBM against the stylized facts and compute a terminal target probability:** Show that GBM growth rates are Gaussian with constant volatility and zero one-step autocorrelation, and derive the closed-form probability that a scheduled long position clears a target return.

Let's get started!
___


## Examples
Today, we will be using the following examples to illustrate key concepts:

> [▶ Build a single asset geometric Brownian motion model](CHEME-5660-L4b-Example-Parameters-SAGBM-Fall-2026.ipynb). Estimate the mean growth rate and volatility of a GBM model from historical price data, recover the drift, simulate sample paths, and compare the model with the observed price trajectory of a firm.

The second example uses the model to evaluate a trade:

> [▶ Implement an NPV-based trade rule using GBM price modeling](CHEME-5660-L4b-Example-GBM-NPV-TradeRule-Fall-2026.ipynb). Compute the probability that a scheduled long position clears a target present-value return when the future share price follows GBM, and plot that probability against the target.
___


## Company Profile: Jane Street
[Jane Street](https://www.janestreet.com) is a privately held quantitative trading firm with approximately 3,000 employees. It trades across asset classes (equities/ETFs, options, bonds, FX, and more) on 200+ venues in 45 countries, and is a major market maker in ETFs. 

> **Business model.** Jane Street (JS) principally deploys its own capital (proprietary trading) while providing liquidity to __institutions__ (e.g., as an ETF authorized participant/lead market maker). The firm is deeply tech-led (functional programming, ML, hardware acceleration) and emphasizes collaborative, firm-wide risk management. [Let's look at what the interns say about Jane Street.](https://www.janestreet.com/join-jane-street/internships/)

**How Jane Street contrasts with Citadel Securities (CS).**
While both Jane Street and Citadel Securities are leading quantitative trading firms, they have some notable differences:

* **Client posture:** [Citadel Securities (CS)](https://www.citadelsecurities.com) handles approximately 40% of U.S. daily retail trade volume for equity/option order flow. On the other hand, Jane Street is primarily proprietary and institution-facing (does handle some retail ETF flow). In June 2025, Citadel Securities handled 38.3% of US retail traffic, while Jane Street handled 11.6%. This data is from [Rule 605 reports](https://www.sec.gov/rule-release/34-43590) filed with the SEC. Here are the [Citadel Securities Rule 605 filings](https://www.citadelsecurities.com/rule-605-606-statements/) and [Jane Street Rule 605 filings](https://www.janestreet.com/execution-quality-reports/). The field descriptions of the Rule 605 reports can be found [here](https://www.sec.gov/files/oes_data-reports-sec_rule_605.pdf).
* **Product emphasis:** Jane Street is especially synonymous with ETFs, while Citadel Securities is most identified with U.S. equities/options/FX market making and order flow.
* **Scale & performance.** Citadel Securities took outside capital [in 2022 at a 22B USD valuation](https://www.reuters.com/business/finance/citadel-securities-valued-22-bln-after-investment-sequoia-paradigm-2022-01-11/). While Jane Street remains privately held with limited external disclosures, it has disclosed larger trading revenue in recent years and exceptionally high trading volumes, particularly in ETFs. Jane Street expanded materially in Asia in 2024–2025 (e.g., Hong Kong).
* JS optimizes for correctness/composability (functional programming [in OCaml](https://ocaml.org), proofs, internal seminars). CS optimizes for industrial reliability at scale (pipelines, hardware, exchange microstructure). Different flavors of intensity.

So it's hard to say which firm is better. Both places are rigorously selective, intensely quantitative, and very tech-forward. From an analytical perspective, here's how one might choose between them:

> __Which should I choose (GPT-5 Thinking):__
> 
> Pick Jane Street if you think: “I’d rather prove properties and build elegant functional programming systems that trade across many venues; I care that my abstractions are beautiful and safe. Optimization axis: JS leans more toward correctness/robustness, model nuance.”
>
> Pick Citadel Securities if you think: “Give me the biggest pipes in the market and the tightest latency/scale constraints; I want my math to survive brutal production realities and move national-level order flow. Optimization axis: CS leans more toward throughput/latency, capacity, reliability under load.”

Citadel Securities skews “hard edges at scale,” Jane Street skews “rigor + collaboration.” Both are intense; they just express intensity differently. Pick your flavor.
___


## Concept Review: N-Ary Tree Models
One obvious limitation of the binomial lattice model is that it only allows for two possible price movements at each time step (up or down). In reality, asset prices can exhibit a wider range of behaviors. 

> __Idea__: If two possible futures are not enough, we can extend the binomial lattice model to an N-ary model, e.g., a trinomial lattice model (three possible price movements: up, down, or unchanged) or even more complex models with multiple price levels at each time step. Let's explore this idea.

Let's do a quick motivational example.

> __Example__
>
> [▶ Let's Explore N-Ary Lattice Models](../L4a/CHEME-5660-L4a-Example-N-Aray-Lattice-Fall-2026.ipynb). In this example, we extend the binomial lattice model to an n-ary lattice model, where the share price can move to many possible values at each time step (not just up or down). In the limit of many possible values, we can approximate a continuous distribution of share prices. Wow!

Wow! That example was cool, and it lays the foundation for today's lecture. With more branches, the set of possible prices at each step gets finer; with a smaller time step, the steps get closer together. That is the idea behind today's lecture: replace the discrete set of possible prices with a continuous one, and the discrete time steps with continuous time. Refinement alone is not enough; with the right scaling of the branch factors and probabilities, the lattice converges to a continuous model, and that model is geometric Brownian motion.
___


## Single Asset Geometric Brownian Motion (GBM)
Geometric Brownian motion (GBM) is a stochastic differential equation describing the share price $S(t)$ as a continuous-time random walk whose drift and noise are both proportional to $S(t)$ (more precisely, as we show below, the log price is a Brownian motion with drift). Over a horizon $0\leq{t}\leq{T}$, the price obeys:
$$
\begin{align*}
\frac{dS\left(t\right)}{S(t)} &= \mu\,{dt}+\sigma\,{dW(t)}\\
\end{align*}
$$
Here, $\mu\in\mathbb{R}$ (units: inverse years) is the arithmetic drift in the GBM price SDE, $\sigma>0$ (units: inverse years to the one-half power) is a constant __volatility__ parameter, and $dW(t)$ is the increment of a Wiener process. We reserve $g$ for an observed continuously compounded growth rate. The drift $\mu$ and a growth rate $g$ share inverse-time units, but they are different quantities; we make the relationship precise below. First, what is a Wiener process?

> __Wiener Process__
> 
> A Wiener Process is a real-valued continuous-time stochastic process named after Norbert Wiener for his study of one-dimensional Brownian motion. A Wiener process is a continuous one-dimensional stochastic process $\left\{W\left(t\right), 0\leq{t}\leq{T}\right\}$ with the following properties:
>
> * The noise vanishes at zero: $W\left(0\right) = 0$ with probability $1$
> * The increments $\left\{W(t_{1}) - W(t_{0}),\dots, W(t_{k}) - W(t_{k-1})\right\}$ are independent for any $k$ and $0\leq{t_{0}}< t_{1} < \dots < t_{k} \leq{T}$
> * The increment $W(t) - W(s)\sim\mathcal{N}\left(0,t-s\right)$ for any $0\leq{s}< t \leq{T}$, where $\mathcal{N}\left(0,t-s\right)$ denotes a normally distributed random variable with mean $0$ and variance $t - s$. Thus, we can write $W(t) - W(s) = \sqrt{t-s}\;Z$, where $Z\sim\mathcal{N}(0,1)$ is a standard normal random variable.

We can solve the [GBM stochastic differential equation using Itô calculus](CHEME-5660-L4b-GBM-Solution-Derivation-Fall-2026.ipynb). One of the really nice features of this model is that it has an analytical solution. Take $t_{0} = 0$ and let $S_{0}$ be the share price today. The share price at a fixed future time $T$ is given by:
$$
\boxed{
\begin{align*}
S_{T} &= S_{0}\;\exp\Biggl[\left(\mu-\frac{\sigma^{2}}{2}\right)T + \sigma\sqrt{T}\;Z\Biggr]\qquad Z\sim\mathcal{N}(0,1)\\
\end{align*}}
$$
Thus, the log price ratio $\ln(S_{T}/S_{0})$ is normally distributed with mean $(\mu-\sigma^{2}/2)T$ and variance $\sigma^{2}T$, and the price $S_{T}$ itself is lognormally distributed and strictly positive. The expectation and variance of $S_{T}$ are given by:
$$
\begin{align*}
\mathbb{E}\left(S_{T}\right) &= S_{0}\;e^{\mu T}\\
\text{Var}\left(S_{T}\right) &= S_{0}^{2}\;e^{2\mu T}\left[e^{\sigma^{2}T} - 1\right]
\end{align*}
$$
For any horizon $T>0$, these expressions show that the expected price grows exponentially at the arithmetic drift $\mu$, while the median price $S_{0}e^{(\mu-\sigma^{2}/2)T}$ grows at the smaller rate $\mu-\sigma^{2}/2$. The gap between the two is the __half-variance correction__: the lognormal distribution is right skewed, so its mean sits above its median.

### Discrete Time GBM Model
The expression for a fixed horizon $T$ is theoretically interesting, but we observe prices at discrete times, e.g., once per trading day. Let's put down a grid of times $t_{j} = j\Delta{t}$ for $j = 0,1,\dots,N$, where $\Delta{t}$ is the time step (units: years) and $T = N\Delta{t}$ is the horizon. Applying the fixed-horizon solution over a single step gives the __one-step transition__:
$$
\boxed{
\begin{align*}
S_{t_{j}} &= S_{t_{j-1}}\;\exp\Biggl[\left(\mu-\frac{\sigma^{2}}{2}\right)\Delta{t} + \sigma\sqrt{\Delta{t}}\;Z_{j}\Biggr]\qquad{j=1,2,\dots,N}\\
\end{align*}}
$$
where the shocks $Z_{1},Z_{2},\dots,Z_{N}$ are __independent__ standard normal random variables (they come from independent Wiener increments). For constant $\mu$ and $\sigma$ this transition is exact at the grid points (it says nothing about the path between them); it is not a numerical time-stepping approximation. Alternatively, we can jump directly from $S_{0}$ to grid point $t_{j}$ using the fixed-horizon solution with $T = t_{j}$:
$$
\begin{align*}
S_{t_{j}} &= S_{0}\;\exp\Biggl[\left(\mu-\frac{\sigma^{2}}{2}\right)t_{j} + \sigma\sqrt{t_{j}}\;\tilde{Z}_{j}\Biggr]\qquad\text{where}\quad\tilde{Z}_{j} = \frac{1}{\sqrt{j}}\sum_{i=1}^{j}Z_{i}\quad{j=1,2,\dots,N}\\
\end{align*}
$$
The cumulative shock $\tilde{Z}_{j}$ is standard normal at each fixed $j$, but $\tilde{Z}_{j}$ and $\tilde{Z}_{k}$ share their first $\min(j,k)$ increments, so they are __not__ independent: their correlation is $\sqrt{t_{\min(j,k)}/t_{\max(j,k)}}$. Use the cumulative form when you need the distribution at one horizon; use the one-step transition, with fresh independent shocks, when you need a price path.

### Monte Carlo Simulation of GBM
We can use the one-step transition to simulate many possible price paths, i.e., possible alternative future prices, using Monte Carlo simulation. 

> __Idea:__ Generate many samples of the independent shocks $Z_{j}$, and use them to generate many possible price paths. We can then analyze the distribution of these price paths to understand the expected price and the uncertainty (variance) in the price.

Let's look at some pseudo-code for the Monte Carlo simulation of the GBM model.

__Initialize:__ Given the initial price $S_{0}$, arithmetic drift $\mu$, volatility $\sigma$, time step $\Delta{t}$, number of steps $N$, and number of sample paths $M$. Initialize an array $\mathbf{S}\in\mathbb{R}^{(N+1)\times M}$ to hold the simulated prices, one path per column, with rows indexed by $j = 0,1,\dots,N$.

For each sample path (each column of $\mathbf{S}$) __do:__
1. Set the price in the first row (grid point $j = 0$) to the initial price $S_{0}$.
2. For each time step $j = 1$ to $N$ __do:__
   - a. Generate a random sample from the standard normal distribution: $Z_{j} \sim \mathcal{N}(0,1)$
   - b. Compute the price at grid point $t_{j}$ from the price at $t_{j-1}$ using the one-step transition:
   $$
   S_{t_{j}} \gets S_{t_{j-1}} \cdot \exp\Biggl[\left(\mu-\frac{\sigma^{2}}{2}\right)\Delta{t} + \sigma\sqrt{\Delta{t}}\;{Z_{j}}\Biggr]
   $$

Every shock is a fresh independent draw, for every step and for every path. This process generates $M$ different price paths (columns) of $N$ steps each, every one a possible future trajectory of the asset price under the GBM model. We've implemented this logic in [the `sample(...)` function](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/equity/#VLQuantitativeFinancePackage.sample-Tuple{MyGeometricBrownianMotionEquityModel,%20NamedTuple}).

___


## Estimation of GBM Parameters
The parameters in the GBM model embody the risk versus return trade-off of the asset. To see how each parameter shows up in data, divide the one-step transition by $S_{t_{j-1}}$, take the natural logarithm, and divide by the time step. This gives the __one-step growth rate__ $g_{j}$ over the interval $t_{j-1}\rightarrow{t_{j}}$, the same continuously compounded growth rate we computed from price data in L3a:
$$
\boxed{
\begin{align*}
g_{j} &\equiv \left(\frac{1}{\Delta{t}}\right)\ln\left(\frac{S_{t_{j}}}{S_{t_{j-1}}}\right) = \underbrace{\left(\mu-\frac{\sigma^{2}}{2}\right)}_{\text{mean growth}\;\bar{g}} + \underbrace{\frac{\sigma}{\sqrt{\Delta{t}}}}_{\text{growth-rate std}}\;Z_{j}\qquad{j=1,2,\dots,N}\\
\end{align*}}
$$
Because the shocks $Z_{j}$ are independent standard normals, the one-step growth rates over equally spaced, non-overlapping steps under constant-parameter GBM are independent and normally distributed with mean $\bar{g}\equiv\mu-\sigma^{2}/2$ and variance $\sigma^{2}/\Delta{t}$. Every estimator in this section falls out of this display. A word on notation: $\bar g$ is the population mean growth rate (a model parameter), $g^{\prime}$ is L3a's symbol for the sample mean of a growth-rate series, and $\widehat{\bar g}$ denotes an estimate of $\bar g$ by whichever method we are discussing.

> __Parameters__
> 
> * __Drift versus growth:__ In $dS/S=\mu\,dt+\sigma\,dW$, $\mu$ is the arithmetic SDE drift. The __mean growth rate__ $\bar{g}=\mathbb{E}[g_{j}]=\mu-\sigma^{2}/2$ (units: inverse years) is what a growth-rate series measures. The two differ by the half-variance correction. Historical estimation makes neither quantity a guaranteed reward.
> * __Risk:__ The __volatility__ parameter $\sigma\in\mathbb{R}_{>0}$ (units: inverse years to the one-half power) sets the standard deviation of the one-step growth rate, $\sigma/\sqrt{\Delta{t}}$, and of the one-step log return $g_{j}\Delta{t}$, which is $\sigma\sqrt{\Delta{t}}$. We can think of volatility as the risk associated with holding the asset. The volatility term is always positive, i.e., $\sigma > 0$.

We estimate $\bar{g}$ and $\sigma$ from a growth-rate series and then recover $\mu$ from the half-variance correction. Let's start with volatility.

### Volatility
Suppose we have price observations $\left\{S_{t_{0}}, S_{t_{1}}, \ldots, S_{t_{N}}\right\}$ on the grid $t_{j} = j\Delta{t}$, so the sample spans $T = N\Delta{t}$ years. From these $N+1$ prices, we construct the $N$ one-step growth rates $\left\{g_{1},g_{2},\ldots,g_{N}\right\}$. The sample mean $g^{\prime}$ and the sample standard deviation $\sigma_{g}$ of the growth-rate series (our risk measure from L3a) are given by:
$$
\begin{align*}
g^{\prime} &= \frac{1}{N}\sum_{j=1}^{N}g_{j}\\
\sigma_{g} &= \underbrace{\sqrt{\frac{1}{N - 1}\sum_{j=1}^{N}\left(g_j - g^{\prime}\right)^2}}_{\text{volatility of the growth rate}} \\
\end{align*}
$$
The denominator $N-1$ is the degrees of freedom of the sample standard deviation: $N$ growth observations less the one mean estimated from them. (In L3a we wrote the same estimator for $T$ price observations; here the sample has $N$ steps so that its span is $T = N\Delta{t}$.) We'll typically compute the standard deviation using [the `std(...)` function exported by the `Statistics.jl` package](https://docs.julialang.org/en/v1/stdlib/Statistics/#Statistics.std).

However, $\sigma_{g}$ is __not__ the GBM volatility $\sigma$. From the growth-rate display, the standard deviation of $g_{j}$ is $\sigma/\sqrt{\Delta{t}}$, so $\sigma_{g}$ estimates $\sigma/\sqrt{\Delta{t}}$, and the volatility estimate is:
$$
\boxed{
\begin{align*}
\hat{\sigma} &= \sigma_{g}\sqrt{\Delta{t}}\\
\end{align*}}
$$
For daily data, $\Delta{t} = 1/252$ years, so the standard deviation of the daily growth rates is divided by $\sqrt{252}$. Equivalently, the standard deviation of the daily log returns $g_{j}\Delta{t}$ is multiplied by $\sqrt{252}$; the companion example computes it that way, and with the same sample standard deviation the two numbers are the same.

### Drift
Next, estimate the mean growth rate $\bar{g}$ (units: inverse years). Under GBM, the arithmetic SDE drift is $\mu=\bar g+\sigma^2/2$, so once we have an estimate $\widehat{\bar g}$ of the mean growth rate, the drift estimate is $\hat{\mu}=\widehat{\bar g}+\hat{\sigma}^2/2$. Both estimates are subject to substantial sampling error; the drift is the harder of the two, because the mean growth is small relative to the growth-rate noise. There are two common ways to estimate $\bar{g}$:

> __Estimating the mean growth rate $\bar g$:__
>
> * __Method 1: Average growth rate.__ Compute the growth rates and take their sample mean, $\widehat{\bar g} = g^{\prime}$. Under GBM this is also the maximum likelihood estimate of $\bar g$.
> * __Method 2: Linear regression.__ Regress the log price on time. The slope estimates $\bar g$, but the residuals are a Brownian path rather than independent noise, so the fit is descriptive: its slope is usable, but the ordinary regression standard errors and residual checks do not apply.
> 
> Let's check out the ideas behind the linear regression method.

#### Linear Regression
Let's assume we have a price dataset for some firm, collected every day for some time period, on the grid $t_{j} = j\Delta{t}$: $\left\{S_{t_{0}}, S_{t_{1}}, \ldots, S_{t_{N}}\right\}$. Taking the natural logarithm of the cumulative GBM solution at grid point $t_{j}$ gives:
$$
\begin{align*}
\ln(S_{t_{j}}) &= \ln(S_{0}) + \bar g\;{t_{j}} + \sigma\;{W(t_{j})}\qquad{j=0,1,\ldots, N}\\
\end{align*}
$$
The expected value of the noise term is zero, so the __expected__ log price is a straight line in time with intercept $\ln(S_{0})$ and slope $\bar g$. That is why regressing the log price on time estimates the mean growth rate: the regression recovers the line, and the Wiener term $\sigma{W(t_{j})}$ plays the role of the error term. Notice what that error term is. It is a Brownian path, so its values at neighboring dates are strongly correlated and its variance $\sigma^{2}t_{j}$ grows along the sample, and the fitted residuals inherit both features. Ordinary least squares still gives a sensible slope, but the textbook standard errors, confidence intervals, and residual-normality checks assume independent, identically distributed errors, and they are not calibrated for correlated, heteroskedastic residuals like these. (The companion example runs an Anderson-Darling normality test on the residuals and it rejects; treat that as a warning about the test's assumptions, not as a verdict on GBM.)

To set up the regression, we construct an overdetermined system of equations from the $N+1$ log prices:
$$
\begin{align*}
\hat{\mathbf{X}}\mathbf{\theta} + \epsilon &= \mathbf{y}\\
\end{align*}
$$
where $\mathbf{\theta}$ contains the model parameters, the intercept $\ln(S_{0})$ and the slope $\bar g$, and $\epsilon$ is the error vector. We estimate the intercept along with the slope, rather than pinning it to the observed $\ln(S_{0})$, so that the fitted line passes through the middle of the data instead of through the first day. The (augmented) design matrix $\hat{\mathbf{X}}$ holds a column of ones for the intercept and the grid times:
$$
\begin{align*}
\hat{\mathbf{X}} &= \begin{bmatrix}
1 & t_{0} \\
1 & t_{1} \\
\vdots & \vdots \\
1 & t_{N}
\end{bmatrix}\\
\end{align*}
$$
while the observation vector $\mathbf{y}$ holds the log prices:
$$
\begin{align*}
\mathbf{y} &= \begin{bmatrix}
\ln(S_{t_{0}}) \\
\ln(S_{t_{1}}) \\
\vdots \\
\ln(S_{t_{N}})
\end{bmatrix}\\
\end{align*}
$$
The parameter estimate $\hat{\mathbf{\theta}}$ (the estimates of the intercept and slope) has an analytical solution given by the normal equations:
$$
\boxed{
\begin{align*}
\hat{\mathbf{\theta}} &= (\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}})^{-1}\hat{\mathbf{X}}^{\top}\mathbf{y}\quad\blacksquare\\
\end{align*}}
$$
The second component of $\hat{\mathbf{\theta}}$ is the regression estimate $\widehat{\bar g}$ of the mean growth rate.

Let's do an example where we estimate the mean growth rate and volatility of a GBM model using historical price data.

> __Example__
> 
> [▶ Build a single asset geometric Brownian motion model](CHEME-5660-L4b-Example-Parameters-SAGBM-Fall-2026.ipynb). In this example, we estimate the mean growth rate and volatility for each firm in a dataset, recover the GBM drift, simulate sample paths, and compare the model with the observed price trajectory of a randomly selected firm.
___


## Stylized Facts
The GBM model is our second model of asset prices. With any model, we want to ensure that it captures the key features of real-world asset prices. Thus, let's ask ourselves: Does the GBM model capture the stylized facts of asset prices?

__Stylized Facts of Asset Prices:__

* __Heavy (also called fat) tailed distribution__: Stock growth rates often exhibit a distribution with heavier tails than would be expected under a normal distribution. This means that extreme price movements are more likely than would be predicted by a normal distribution.
* __Absence of Autocorrelation__: Autocorrelation refers to the tendency of growth rates to correlate with their past values. Autocorrelation suggests predictability in growth rates, which traders could exploit. On the other hand, if prices follow a random walk, then the growth rates over non-overlapping intervals are uncorrelated.
* __Volatility clustering__: Stock growth rates tend to be more volatile during specific periods and less volatile during others. This phenomenon is known as volatility clustering, suggesting that large price movements are more likely to be followed by other large moves, and other small moves follow small moves.

We already have the distribution of the one-step growth rate $g_{j}$ from the estimation section: it is normal with mean $\bar g$ and variance $\sigma^{2}/\Delta{t}$. Let's also look at growth over a longer horizon. Define the __horizon-average growth rate__ over $0\rightarrow{T}$ as $g_{T,0}\equiv(1/T)\ln(S_{T}/S_{0})$, with the same to-from subscript order as the discount factor $\mathcal{D}_{T,0}$. Substituting the GBM solution gives:
$$
\begin{align*}
\ln\left(\frac{S_{T}}{S_{0}}\right) &= \left(\mu-\frac{\sigma^{2}}{2}\right)T + \sigma\sqrt{T}\;Z\quad\Longrightarrow\text{divide by the horizon}\;T\\
\underbrace{\left(\frac{1}{T}\right)\ln\left(\frac{S_{T}}{S_{0}}\right)}_{= g_{T,0}} &= \bar g + \frac{\sigma}{\sqrt{T}}\;Z\\
\end{align*}
$$
Putting this all together, the horizon-average growth rate under GBM is:
$$
\boxed{
\begin{align*}
g_{T,0} &= \bar g + \frac{\sigma}{\sqrt{T}}\;Z\qquad Z\sim\mathcal{N}(0,1)\\
\end{align*}}
$$
Thus, $g_{T,0}$ is normally distributed with mean $\bar g$ and variance $\sigma^{2}/T$; the one-step case is $T = \Delta{t}$. The uncertainty of the horizon-average growth rate decreases as the horizon increases, while its mean does not move. Over a long enough window the average growth rate is pinned near $\bar g$, but the price is not: the log price ratio $\ln(S_{T}/S_{0}) = T\,g_{T,0}$ has variance $\sigma^{2}T$, which grows with $T$.

Wow! This already gives us some clues about whether the GBM model captures the stylized facts of asset prices.

> __Heavy-Tailed Growth Rates:__ Both one-step and horizon-average growth rates are normally distributed under GBM. Thus, the GBM model __will not capture__ the heavy-tailed nature of empirical asset growth rates. (The lognormal price distribution has a long right tail, but that is not the same thing as a heavy-tailed growth-rate distribution.)

Volatility clustering is ruled out by construction:

> __Volatility Clustering:__ The GBM volatility parameter $\sigma$ is constant. Thus, there is no mechanism in the model to capture volatility clustering, i.e., the alternation between high- and low-volatility periods.

Finally, let's consider the autocorrelation of the one-step growth rates. From the estimation section, $g_{j} = \bar g + (\sigma/\sqrt{\Delta{t}})\,Z_{j}$ with independent shocks $Z_{j}$. Then, the autocovariance at lag $\tau\ge 1$ is given by:
$$
\begin{align*}
\mathrm{Cov}(g_{j},g_{j+\tau})
= \mathrm{Cov}\!\left(\bar g+\frac{\sigma}{\sqrt{\Delta{t}}}Z_{j},\;\bar g+\frac{\sigma}{\sqrt{\Delta{t}}}Z_{j+\tau}\right)
= \frac{\sigma^{2}}{\Delta{t}}\,\mathrm{Cov}(Z_{j},Z_{j+\tau})
= 0,
\end{align*}
$$
because $Z_{j}$ and $Z_{j+\tau}$ are independent (from our definition of the noise process). Hence the autocorrelation function is given by:
$$
\begin{align*}
\rho_{g}(\tau)=\frac{\mathrm{Cov}(g_{j},g_{j+\tau})}{\mathrm{Var}(g_{j})}=0,\qquad \tau\ge 1,
\end{align*}
$$
with $\mathrm{Var}(g_{j})=\sigma^2/\Delta t$.

> __Absence of Autocorrelation:__ The autocorrelation function $\rho_{g}(\tau)$ is zero for all lags $\tau\ge 1$. Thus, the GBM model __does capture__ the absence of autocorrelation in one-step growth rates over non-overlapping intervals. So, the GBM model captures this stylized fact! It says nothing, however, about the squared growth rates, whose dependence we saw in L3a and which GBM also cannot produce.

___


## GBM Trade Rule
Let's develop an expression that allows us to compute the probability that a trade clears a target return, based on the net present value (NPV) of the trade, when the share price follows GBM.

> __Scenario:__ Suppose we purchase $n_{0}>0$ shares of ticker `XYZ` at time $t=0$ (today) for $S_{0}$ USD/share.
> Then, sometime later, at $T=N\Delta{t}$, we sell all $n_{0}$ shares at a share price of $S_{T}$ USD/share, where $N\geq{1}$ is the number of time steps (integer) and $\Delta{t}$ is the time step (e.g., one trading day written in units of years).

This trade is a __long position__ because we profit if the share price goes up, i.e., $S_{T} > S_{0}$. As in L4a, we work in a frictionless baseline: no trading costs, and a price return with no dividends. Further, we can model the trade as an __abstract asset__ with two cash flow events: the initial purchase at $t=0$ (entry) and the sale at $t=T$ (exit). The net present value (NPV) of this trade (written from our perspective) is:
$$
\begin{align*}
\texttt{NPV}(g_b, T) &= \underbrace{-n_{0}\;{S_{0}}}_{\text{entry (now)}} + \underbrace{n_{0}\;{S_{T}}\;\mathcal{D}_{T,0}^{-1}(g_b)}_{\text{exit (future)}}\quad\Longrightarrow\text{divide by initial investment}\;n_{0}\;{S_{0}}\\
\frac{\texttt{NPV}(g_b, T)}{n_{0}\;{S_{0}}} &= \frac{S_{T}\;\mathcal{D}_{T,0}^{-1}(g_b) - S_{0}}{S_{0}}\\
\underbrace{\frac{\texttt{NPV}(g_b, T)}{n_{0}\;{S_{0}}}}_{\text{fractional return}\;\rho_{T}} &= \left(\frac{S_{T}}{S_{0}}\right)\;\mathcal{D}_{T,0}^{-1}(g_b) - 1\quad\blacksquare
\end{align*}
$$
Here $g_b$ is the continuously compounded annual benchmark growth rate from L4a, $\mathcal{D}_{T,0}(g_b)=e^{g_bT}$ is the accumulation factor, and $\mathcal{D}^{-1}_{T,0}(g_b)=e^{-g_bT}$ discounts the terminal sale value to time 0. As in L4a, we call the left-hand side, the NPV per dollar invested, the scaled NPV $\rho_{T}$; it is a dimensionless present-value return on the initial share cost.

Last lecture, we used a binomial lattice model for $S_{T}$. Let's now assume that the share price $S_{T}$ follows a GBM model. Substituting the GBM solution for $S_{T}$ into the scaled NPV expression gives:
$$
\boxed{
\begin{align*}
\rho_{T} = \frac{\texttt{NPV}(g_b, T)}{n_{0}{S_{0}}} &= \exp\Biggl[\left(\mu-\frac{\sigma^{2}}{2}\right)T + \sigma\sqrt{T}\;Z\Biggr]\;e^{-g_bT} - 1\qquad Z\sim\mathcal{N}(0,1)\quad\blacksquare\\
\end{align*}}
$$
If we hold the $n_{0}$ shares for only a __short time__, e.g., a few days or a few weeks, then for realistic choices of the benchmark rate $|g_b|T$ is small, $e^{-g_bT}\approx1$, and $\rho_{T}\approx S_{T}/S_{0}-1$ is approximately the fractional change in share price. For a longer holding period, or whenever $|g_b|T$ is not small, keep the discount factor. Either way, $\rho_{T}$ inherits its randomness from the single standard normal $Z$, which is what makes the probability calculation below a one-line standardization.


### Cumulative Probability of a Trade Exit
Unlike the binomial lattice model from last lecture, where $\rho_{T}$ took one of $N+1$ values, the scaled NPV is now a __continuous random variable__, because the share price $S_{T}$ is. However, we can still compute the probability that our trade meets a desired fractional return target, $\rho_{T} > \rho_{\star}$, at the scheduled exit $T$, given the benchmark rate $g_b$. As in L4a, the event is the scheduled terminal exit; a take-profit or stop-loss rule monitored before $T$ is a different (first-passage) question.

Let's dig into this a bit more. Let $\rho_{\star}>-1$ be the target. Because $S_{T}>0$ under GBM, the scaled NPV is strictly greater than $-1$, so this is the whole feasible range; a target of $-1$ or below is cleared with probability one. We want the probability that $\rho_{T}$ is (strictly) greater than $\rho_{\star}$. Substituting the boxed expression for $\rho_{T}$, and moving everything that is not random to the right-hand side, gives:
$$
\begin{align*}
\mathbb{P}\!\left(\rho_{T}> \rho_\star\right)
& = \mathbb{P}\!\left(\left(\frac{S_{T}}{S_{0}}\right)e^{-g_bT}-1> \rho_\star\right)\\
& = \mathbb{P}\!\left(\ln\left(\frac{S_{T}}{S_{0}}\right)>\ln(1+\rho_\star) + g_bT\right)
\end{align*}
$$
Ok, so here is the trick to this type of probability calculation. We need to write the probability expression in terms of the cumulative distribution function (CDF) of the standard normal random variable $Z$. To do this, we standardize the log price ratio, which under GBM is normal with mean $(\mu-\sigma^{2}/2)T$ and standard deviation $\sigma\sqrt{T}$ (positive, since $\sigma>0$ and $T>0$). Subtracting the mean and dividing by the standard deviation on both sides of the inequality gives:
$$
\begin{align*}
\mathbb{P}\!\left(\rho_{T}> \rho_\star\right)
= \mathbb{P}\!\left(\underbrace{\frac{\ln(S_{T}/S_{0})-(\mu-\sigma^{2}/2)T}{\sigma\sqrt{T}}}_{Z\sim\mathcal{N}(0,1)}>\frac{\ln(1+\rho_\star)+g_bT-(\mu-\sigma^{2}/2)T}{\sigma\sqrt{T}}\right)
\end{align*}
$$
The probability that a standard normal random variable exceeds a number $z$ is $1-\Phi(z)$, where $\Phi(\cdot)$ is the CDF of a standard normal random variable. Thus, the probability that the trade clears the target is:
$$
\boxed{\;
\mathbb{P}\!\left(\rho_{T}> \rho_\star\right)
= 1 - \Phi\!\left(\frac{\ln(1+\rho_\star)+g_bT-\left(\mu-\frac{\sigma^{2}}{2}\right)T}{\sigma\sqrt{T}}\right)\quad\blacksquare}
$$
This probability uses the real-world drift $\mu$ and volatility $\sigma$ estimated from data. Replacing $\mu$ by the risk-neutral drift, as we will do when we price derivatives later in the course, changes the measure and no longer answers a probability-of-profit question. For a short holding period, $g_bT\approx0$ and the benchmark drops out of the numerator.

Let's do an example.

> __Example__
>
> [▶ Implement an NPV-based trade rule using GBM price modeling](CHEME-5660-L4b-Example-GBM-NPV-TradeRule-Fall-2026.ipynb). In this example, we evaluate the probability that a scheduled long position clears a target return when the future share price is modeled using GBM, and plot the probability against the target.

___


## Optional Advanced Material
The notebooks below extend today's material. They are optional and are not prerequisites for L5a; the [advanced index](advanced/README.md) lists them with a suggested order.

* [▶ From the binomial lattice to GBM](advanced/lattice-limit/CHEME-5660-L4b-Advanced-LatticeToGBM-Fall-2026.ipynb). Calibrate the binomial lattice of L3b and L4a to the GBM drift and volatility, and watch the terminal distribution and the L4a binomial-tail target probability converge to the lognormal and to today's closed form.
* [▶ First-passage rules under GBM](advanced/first-passage/CHEME-5660-L4b-Advanced-FirstPassage-GBM-Fall-2026.ipynb). Compute take-profit and stop-loss first-passage probabilities under GBM (a closed form for one barrier, the L4a absorbing recursion and Monte Carlo for two) and quantify the effect of the monitoring frequency.
* [▶ Drift uncertainty](advanced/drift-uncertainty/CHEME-5660-L4b-Advanced-DriftUncertainty-Fall-2026.ipynb). Show that the drift estimate depends on the calendar span of the data and not on the sampling frequency, and propagate that uncertainty into the target probability.
* [▶ Monte Carlo versus the closed form](advanced/monte-carlo/CHEME-5660-L4b-Advanced-MonteCarlo-TargetProbability-Fall-2026.ipynb). Estimate the target probability by simulation with standard errors, compare the exact one-step transition with the Euler scheme, and reduce variance with antithetic variates.
___

## Summary
In this lecture, we introduced geometric Brownian motion as our first continuous-time model of the share price, estimated its parameters from growth-rate data, tested it against the stylized facts, and derived a closed-form probability for a scheduled trade.

> __Key Takeaways:__
>
> * **GBM is a tractable lognormal price model:** The stochastic differential equation has an exact solution in which the log price ratio is normal, so the price is lognormal, its mean grows at the arithmetic drift, and price paths can be simulated exactly one step at a time with independent standard normal shocks.
>
> * **Drift and growth are different parameters, and both come from the growth-rate series:** The one-step growth rate is normal with mean equal to the drift less half the variance, and with standard deviation equal to the volatility divided by the square root of the time step; therefore the sample mean estimates the mean growth rate, the sample standard deviation scaled by the square root of the time step estimates the volatility, and the arithmetic drift follows from the half-variance correction. Regression on log price also estimates the mean growth rate, but its residuals are a Brownian path, so its standard errors do not apply.
>
> * **GBM passes one stylized fact, fails two, and gives a closed-form target probability:** One-step growth rates are uncorrelated under GBM, but they are Gaussian with constant volatility, so the model cannot produce heavy tails or volatility clustering; the probability that a scheduled long position clears a target return is one minus a standard normal CDF evaluated at a standardized threshold.

Next time, we extend GBM to several correlated assets and use it to build portfolios.
___


## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___